In [0]:
%sql
-- Email Masking: john.smith@examplebank.com → j***@examplebank.com
CREATE OR REPLACE FUNCTION mask_email(email STRING)
RETURNS STRING
RETURN 
  CASE 
    WHEN email IS NULL THEN NULL
    ELSE CONCAT(
      LEFT(email, 1),                              -- first character
      '***',                                       -- mask separator
      SUBSTRING(email, INSTR(email, '@'))          -- @domain.com
    )
  END;

-- Phone Masking: 9876543210 → *******3210
CREATE OR REPLACE FUNCTION mask_phone(phone STRING)
RETURNS STRING
RETURN
  CASE
    WHEN phone IS NULL THEN NULL
    ELSE CONCAT(
      REPEAT('*', LENGTH(phone) - 4),              -- stars for all but last 4
      RIGHT(phone, 4)                              -- last 4 digits
    )
  END;

In [0]:
%sql
-- View 1: AUTHORIZED users see real data
CREATE OR REPLACE VIEW digital_banking.gold.silver_customer_authorized AS
SELECT * FROM digital_banking.silver.silver_customers;

-- View 2: UNAUTHORIZED users see masked PII
CREATE OR REPLACE VIEW digital_banking.gold.silver_customer_public AS
SELECT 
  customer_id,
  first_name,
  last_name,
  date_of_birth,
  mask_email(email)   AS email,
  mask_phone(phone)   AS phone,
  address,
  city,
  state,
  postal_code,
  customer_segment,
  customer_status,
  registration_date,
  updated_at
FROM digital_banking.silver.silver_customers;

In [0]:
%sql
SELECT * FROM digital_banking.gold.silver_customer_public